In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Let's start with necessary imports
import os
import numpy as np
import pandas as pd
from shutil import copyfile
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt

## Define Settings

In [3]:
data_dir = "../../data/images/"
meta_data_path = "../../data/butterfly_anomaly_train.csv"
plot_dir = "../../plots/"
model_dir = "../../data/models/"

In [4]:
for dir_name in [plot_dir, model_dir]:
    if not os.path.exists(dir_name):
        os.makedirs(dir_name)

## Define Data Loader

In [ ]:
from hdr_hybrid_butterflies.data_handler import DataHandler, ImageProcessor

data_handler = DataHandler(
    meta_data_path=meta_data_path,
    data_dir=data_dir,
)

In [ ]:
data_handler.df_meta.iloc[10]["hybrid_stat"] == "nonhybrid"

In [ ]:
data_handler.df_meta.iloc[10]

In [ ]:
data_handler.load_data(1804)[0]

### Copy Files of Subspecies to sub-directories

In [ ]:
np.unique(data_handler.df_meta["subspecies"], return_counts=True)

In [10]:
if False:
    for idx, row in tqdm(
        data_handler.df_meta.iterrows(), total=data_handler.n_samples
    ):
        if row["hybrid_stat"] == "non-hybrid":
            input_path = os.path.join(
                data_dir, row["hybrid_stat"], row["filename"]
            )
            output_path = os.path.join(
                data_dir,
                row["hybrid_stat"],
                f"{int(row['subspecies']):02d}",
                row["filename"],
            )
            output_dir = os.path.dirname(output_path)
            if not os.path.exists(output_dir):
                os.makedirs(output_dir)
            copyfile(input_path, output_path)

## Create overview of all Subspecies

In [ ]:
n_subspecies = len(np.unique(data_handler.df_meta["subspecies"]))
n_subspecies

In [ ]:
data_handler.df_meta[
    data_handler.df_meta["subspecies"].isnull()
].parent_subspecies_2.unique()

In [13]:
if False:
    for seed in range(10):
        rng = np.random.default_rng(seed)

        sub_species = np.unique(data_handler.df_meta["subspecies"])

        fig, axes = plt.subplots(3, 5, figsize=(30, 15))
        axes_flat = axes.flatten()

        for idx, sub in enumerate(sub_species):
            if np.isnan(sub):
                indices = data_handler.df_meta[
                    data_handler.df_meta["subspecies"].isnull()
                ].index
            else:
                indices = data_handler.df_meta[
                    data_handler.df_meta["subspecies"] == sub
                ].index

            chosen_idx = rng.choice(indices)

            img, row = data_handler.load_data(chosen_idx)
            axes_flat[idx].imshow(img)
            axes_flat[idx].set_title(
                f"Subspecies: {row['subspecies']} | Idx: {chosen_idx} | Occurrences: {len(indices)}"
            )
            axes_flat[idx].axis("off")

        plt.tight_layout()
        fig.savefig(
            os.path.join(plot_dir, f"subspecies_examples_{seed:04d}.png")
        )

## Test Grounded Segment Anything (Grounded DINO + SAM)

In [ ]:
image, row = data_handler.load_data(874)
image

In [ ]:
from PIL import ImageOps

# image, row = data_handler.load_by_name("CAM008547.jpg")
# image, row = data_handler.load_by_name("CAM011441")
image, row = data_handler.load_by_name("CAM036588")
image = ImageOps.exif_transpose(image)
image

In [16]:
from hdr_hybrid_butterflies.dino_sam.dino_sam import grounded_segmentation

In [ ]:
1237
# labels = ["upper left wing.", "lower left wing.", "upper right wing.", "lower right wing."]
labels = [
    "upper left butterfly wing.",
    "lower left butterfly wing.",
    "upper right butterfly wing.",
    "lower right butterfly wing.",
]
# labels = ["upper wing.", "lower wing."]
# labels = ["left wing.", "right wing."]
labels = ["wings."]
# labels = ["upper wing.", "lower wing."]
# labels = ["larger wing.", "smaller wing."]
# labels = ["butterfly wing."]
threshold = 0.2

detector_id = "IDEA-Research/grounding-dino-tiny"
segmenter_id = "facebook/sam-vit-base"


image_array, detections = grounded_segmentation(
    image=image,
    labels=labels,
    threshold=threshold,
    polygon_refinement=True,
    detector_id=detector_id,
    segmenter_id=segmenter_id,
)

In [18]:
from hdr_hybrid_butterflies.dino_sam import plotting

In [ ]:
def get_most_confident_detection(detections, label):
    detections_mask = [d for d in detections if d.label == label]
    return sorted(detections_mask, key=lambda x: x.score)[-1]


detections_confident = [
    get_most_confident_detection(detections, label) for label in labels
]
detections_confident

In [ ]:
fig, ax = plotting.plot_detections(image_array, detections)
fig.savefig(os.path.join(plot_dir, "detections.png"))

In [ ]:
fig, ax = plotting.plot_detections(image_array, detections_confident)

## Create training data for segment classifier 

Classified created image segments into one of the following classes: 
    upper / lower / noise
Further steps will only be run on segments that pass the classifier

In [22]:
segment_training_dir = os.path.join(data_dir, "segment_classier_training")

if not os.path.exists(segment_training_dir):
    os.makedirs(segment_training_dir)

In [ ]:
from PIL import Image
from glob import glob

data_processor = ImageProcessor()

for idx, row in tqdm(
    data_handler.df_meta.iterrows(), total=data_handler.n_samples
):
    file_glob = f"{row['filename'][:-4]}_*.jpg"
    if len(glob(os.path.join(segment_training_dir, "all", file_glob))) > 0:
        print(f"Skipping {row['filename']}")
        continue

    image, row = data_handler.load_data(idx)
    segments, scores = data_processor._raw_segments(image)

    for idx_j, segment in enumerate(segments):
        score = scores[idx_j]
        filename = f"{row['filename'][:-4]}_{idx_j:04d}_s{score:0.4f}.jpg"
        PIL_image = Image.fromarray(segment)
        PIL_image.save(os.path.join(segment_training_dir, "all", filename))

Re-create segments for individual input files

In [24]:
if False:
    input_file_glob = "../../data/images/non-hybrid/11/*.jpg"
    segment_individual_dir = os.path.join(segment_training_dir, "individual")

    if not os.path.exists(segment_individual_dir):
        os.makedirs(segment_individual_dir)

    input_files = sorted(glob(input_file_glob))
    for input_file in tqdm(input_files, total=len(input_files)):
        image = Image.open(input_file)
        image = ImageOps.exif_transpose(image)

        camid = os.path.basename(input_file)[:-4]

        segments, scores = data_processor._raw_segments(image)

        for idx_j, segment in enumerate(segments):
            score = scores[idx_j]
            filename = f"{camid}_{idx_j:04d}_s{score:0.4f}.jpg"
            PIL_image = Image.fromarray(segment)
            PIL_image.save(os.path.join(segment_individual_dir, filename))

Separate segments out into upper, lower, and noise directories

In [25]:
upper_wing_dir = os.path.join(segment_training_dir, "upper_wing")
lower_wing_dir = os.path.join(segment_training_dir, "lower_wing")
noise_dir = os.path.join(segment_training_dir, "noise")

for output_dir in [upper_wing_dir, lower_wing_dir, noise_dir]:
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

In [26]:
import imagesize

if False:
    segment_files = sorted(
        glob(os.path.join(segment_training_dir, "all", "*.jpg"))
    )

    for idx, filename in enumerate(segment_files):
        width, height = imagesize.get(filename)
        ratio = width / height
        score = float(filename.split("_")[-1][1:-4])
        if score > 0.5:
            if ratio > 1.4:
                copyfile(
                    filename,
                    os.path.join(upper_wing_dir, os.path.basename(filename)),
                )
            else:
                copyfile(
                    filename,
                    os.path.join(lower_wing_dir, os.path.basename(filename)),
                )
        elif score > 0.1:
            copyfile(
                filename, os.path.join(noise_dir, os.path.basename(filename))
            )

        print(f"{idx:04d} | {ratio:0.3f} | {width}x{height} | {score:0.4f}")

    len(segment_files)

In [ ]:
from hdr_hybrid_butterflies.data_handler import (
    SegmentDataHandler,
    ImageProcessor,
)

image_processor = ImageProcessor()

segment_data_handler = SegmentDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    data_dir_noise=os.path.join(
        segment_training_dir, "manual", "noise_manual"
    ),
    image_processor=image_processor,
)
segment, segment_info = segment_data_handler.load_data(0)
segment

In [28]:
segment_generator_test = segment_data_handler.get_generator(
    batch_size=16,
    queue_size=160,  # 1600,
    n_jobs=1,
    mask_only=True,
    training=False,
)

In [ ]:
segments, segment_labels = next(segment_generator_test)
fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes_flat = axes.flatten()
for idx, ax in enumerate(axes_flat):
    ax.imshow(segments[idx])
    ax.set_title(f"Label: {segment_labels[idx]}")
    ax.axis("off")

In [ ]:
%timeit segment_data_handler()

## Train Segment Classifier

In [30]:
segment_generator_train = segment_data_handler.get_generator(
    batch_size=16,
    queue_size=10000,
    n_jobs=12,
    mask_only=True,
    training=True,
)

In [ ]:
image_processor.output_dim

In [ ]:
import tensorflow as tf
from hdr_hybrid_butterflies.model import CNNClasifier

model = CNNClasifier(
    image_size=image_processor.output_dim,
    num_classes=3,
)
print(model.call(segments).shape)
model.summary()

In [33]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

In [34]:
segment_model_dir = os.path.join(model_dir, "segment_model")

In [ ]:
model.load_weights(os.path.join(segment_model_dir, "model.weights.h5"))

In [36]:
# Create the ModelCheckpoint callback
checkpoint_path = os.path.join(segment_model_dir, "model.weights.h5")

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    monitor="loss",  # 'val_loss'
    mode="min",
    save_best_only=False,
    save_freq="epoch",  # Save every epoch
    verbose=1,
)

In [ ]:
steps_per_epoch = 100
epochs = 100


model.fit(
    segment_generator_train,
    validation_data=segment_generator_test,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_steps=5,
    callbacks=[model_checkpoint_callback],
)

#### Evaluate Segmentation Model

In [ ]:
segments, segment_labels = next(segment_generator_test)

# get prediction
logits = model(segments)
probs = model.logits2probs(logits)

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes_flat = axes.flatten()
for idx, ax in enumerate(axes_flat):
    ax.imshow(segments[idx])
    ax.set_title(
        f"Label: {segment_labels[idx]} | Pred: {np.argmax(probs[idx])} ["
        f"{probs[idx][0]:0.2f}, {probs[idx][1]:0.2f}, {probs[idx][2]:0.2f}]"
    )
    ax.axis("off")

In [43]:
%matplotlib inline

In [ ]:
from sklearn.metrics import classification_report

n_steps = 100

incorrect_segments = []
incorrect_labels = []

predictions = []
labels = []
for idx in tqdm(range(n_steps), total=n_steps):
    segments, segment_labels = next(segment_generator_test)
    prediction = np.argmax(model.logits2probs(model(segments)), axis=-1)

    mask = prediction != segment_labels
    if np.any(mask):
        incorrect_segments.append(segments[mask])
        incorrect_labels.append(segment_labels[mask])

    predictions.append(prediction)
    labels.append(segment_labels)

predictions = np.concatenate(predictions)
labels = np.concatenate(labels)

print(classification_report(labels, predictions))

incorrect_labels = np.concatenate(incorrect_labels)
incorrect_segments = np.concatenate(incorrect_segments)

In [ ]:
predictions = model.logits2probs(model(incorrect_segments))
for label, segment, prediction in zip(
    incorrect_labels, incorrect_segments, predictions
):
    plt.imshow(segment)
    plt.title(
        f"Label: {label} | Pred: {np.argmax(prediction)} "
        f"[{prediction[0]:0.2f}, {prediction[1]:0.2f}, {prediction[2]:0.2f}]"
    )
    plt.axis("off")
    plt.show()

In [ ]:
%timeit model(segments)

In [ ]:
del segment_data_handler
del segment_generator_train
del segment_generator_test

# garbage collect
import gc

gc.collect()

## Train Lower Wing Classifier

In [30]:
feature_number = 3

In [31]:
from hdr_hybrid_butterflies.data_handler import LowerWingDataHandler

lower_wing_data_handler = LowerWingDataHandler(
    meta_data_path=meta_data_path,
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    image_processor=image_processor,
)

In [ ]:
lower_wing_data_handler.n_samples

In [33]:
lower_wing_generator_train = lower_wing_data_handler.get_generator(
    batch_size=16,
    queue_size=10000,
    n_jobs=12,
    mask_only=False,
    training=True,
    labels_func_name=f"labels_feature_{feature_number:02d}",
)

lower_wing_generator_test = lower_wing_data_handler.get_generator(
    batch_size=16,
    queue_size=500,
    n_jobs=1,
    mask_only=False,
    training=False,
    labels_func_name=f"labels_feature_{feature_number:02d}",
)

In [ ]:
import tensorflow as tf
from hdr_hybrid_butterflies.model import CNNClasifier

model_lower = CNNClasifier(
    image_size=image_processor.output_dim,
    num_classes=2,
    name=f"lower_wing_feature_{feature_number:02d}",
)
print(model_lower.call(segments).shape)
model_lower.summary()

In [35]:
model_lower.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

In [36]:
lower_model_dir = os.path.join(model_dir, f"lower_model_{feature_number:02d}")

In [ ]:
model_lower.load_weights(os.path.join(lower_model_dir, "model.weights.h5"))

In [38]:
# Create the ModelCheckpoint callback
checkpoint_path_lower = os.path.join(lower_model_dir, "model.weights.h5")

lower_model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path_lower,
    save_weights_only=True,
    monitor="loss",  # 'val_loss'
    mode="min",
    save_best_only=False,
    save_freq="epoch",  # Save every epoch
    verbose=1,
)

In [ ]:
steps_per_epoch = 100
epochs = 100


model_lower.fit(
    lower_wing_generator_train,
    validation_data=lower_wing_generator_test,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_steps=5,
    callbacks=[lower_model_checkpoint_callback],
)

#### Evaluate Lower Wing Model

In [40]:
%matplotlib inline

In [ ]:
segments, segment_labels = next(lower_wing_generator_test)

# get prediction
logits = model_lower(segments)
probs = model_lower.logits2probs(logits)

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes_flat = axes.flatten()
for idx, ax in enumerate(axes_flat):
    ax.imshow(segments[idx])
    ax.set_title(
        f"Label: {segment_labels[idx]} | Pred: {np.argmax(probs[idx])} ["
        f"{probs[idx][0]:0.2f}, {probs[idx][1]:0.2f}]"
    )

    # red frame around axes if prediction is wrong
    frame_color = (
        "green" if np.argmax(probs[idx]) == segment_labels[idx] else "red"
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(frame_color)
        spine.set_linewidth(7)

    # turn off everything except the spine (i.e. frame)
    ax.tick_params(
        axis="both",
        which="both",
        left=False,
        right=False,
        bottom=False,
        top=False,
        labelleft=False,
        labelbottom=False,
    )

In [ ]:
from sklearn.metrics import classification_report

n_steps = 100

incorrect_segments = []
incorrect_labels = []

predictions = []
labels = []
for idx in tqdm(range(n_steps), total=n_steps):
    segments, segment_labels = next(lower_wing_generator_test)
    prediction = np.argmax(
        model_lower.logits2probs(model_lower(segments)), axis=-1
    )

    mask = prediction != segment_labels
    if np.any(mask):
        incorrect_segments.append(segments[mask])
        incorrect_labels.append(segment_labels[mask])

    predictions.append(prediction)
    labels.append(segment_labels)

predictions = np.concatenate(predictions)
labels = np.concatenate(labels)

print(classification_report(labels, predictions))

incorrect_labels = np.concatenate(incorrect_labels)
incorrect_segments = np.concatenate(incorrect_segments)

In [ ]:
predictions = model_lower.logits2probs(model_lower(incorrect_segments))
for label, segment, prediction in zip(
    incorrect_labels, incorrect_segments, predictions
):
    plt.imshow(segment)
    plt.title(
        f"Label: {label} | Pred: {np.argmax(prediction)} "
        f"[{prediction[0]:0.2f}, {prediction[1]:0.2f}]"
    )
    plt.axis("off")
    plt.show()

## Train Upper Wing Classifier

In [41]:
feature_number = 10

In [42]:
from hdr_hybrid_butterflies.data_handler import UpperWingDataHandler

upper_wing_data_handler = UpperWingDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    image_processor=image_processor,
    # test_split=0.05,
)

In [ ]:
upper_wing_data_handler.n_samples

In [ ]:
result = np.unique(
    upper_wing_data_handler.df_meta["subspecies"].iloc[
        : upper_wing_data_handler.n_samples_train
    ],
    return_counts=True,
)
for sub, count in zip(*result):
    print(f"Subspecies: {sub} | Count: {count}")

In [ ]:
upper_wing_generator_train = upper_wing_data_handler.get_generator(
    batch_size=16,
    queue_size=10000,
    n_jobs=12,
    mask_only=False,
    training=True,
    balanced_loading=True,
    labels_func_name=f"labels_feature_{feature_number:02d}",
)

upper_wing_generator_test = upper_wing_data_handler.get_generator(
    batch_size=16,
    queue_size=1600,
    n_jobs=1,
    mask_only=False,
    training=False,
    labels_func_name=f"labels_feature_{feature_number:02d}",
)

In [ ]:
import tensorflow as tf
from hdr_hybrid_butterflies.model import CNNClasifier

model_upper = CNNClasifier(
    image_size=image_processor.output_dim,
    num_classes=2,
    name=f"upper_wing_feature_{feature_number:02d}",
)
print(model_upper.call(segments).shape)
model_upper.summary()

In [47]:
model_upper.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

In [48]:
upper_model_dir = os.path.join(model_dir, f"upper_model_{feature_number:02d}")

In [ ]:
model_upper.load_weights(os.path.join(upper_model_dir, "model.weights.h5"))

In [50]:
# Create the ModelCheckpoint callback
checkpoint_path_upper = os.path.join(upper_model_dir, "model.weights.h5")

upper_model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path_upper,
    save_weights_only=True,
    monitor="loss",  # 'val_loss'
    mode="min",
    save_best_only=False,
    save_freq="epoch",  # Save every epoch
    verbose=1,
)

In [ ]:
steps_per_epoch = 100
epochs = 100


model_upper.fit(
    upper_wing_generator_train,
    validation_data=upper_wing_generator_test,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_steps=5,
    callbacks=[upper_model_checkpoint_callback],
)

#### Evaluate Upper Wing Model

In [45]:
%matplotlib inline

In [ ]:
segments, segment_labels = next(upper_wing_generator_test)

# get prediction
logits = model_upper(segments)
probs = model_upper.logits2probs(logits)

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes_flat = axes.flatten()
for idx, ax in enumerate(axes_flat):
    ax.imshow(segments[idx])
    ax.set_title(
        f"Label: {segment_labels[idx]} | Pred: {np.argmax(probs[idx])} ["
        f"{probs[idx][0]:0.2f}, {probs[idx][1]:0.2f}]"
    )

    # red frame around axes if prediction is wrong
    frame_color = (
        "green" if np.argmax(probs[idx]) == segment_labels[idx] else "red"
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(frame_color)
        spine.set_linewidth(7)

    # turn off everything except the spine (i.e. frame)
    ax.tick_params(
        axis="both",
        which="both",
        left=False,
        right=False,
        bottom=False,
        top=False,
        labelleft=False,
        labelbottom=False,
    )

In [ ]:
from sklearn.metrics import classification_report

n_steps = 100

incorrect_segments = []
incorrect_labels = []

predictions = []
labels = []
for idx in tqdm(range(n_steps), total=n_steps):
    segments, segment_labels = next(upper_wing_generator_test)
    prediction = np.argmax(
        model_upper.logits2probs(model_upper(segments)), axis=-1
    )

    mask = prediction != segment_labels
    if np.any(mask):
        incorrect_segments.append(segments[mask])
        incorrect_labels.append(segment_labels[mask])

    predictions.append(prediction)
    labels.append(segment_labels)

predictions = np.concatenate(predictions)
labels = np.concatenate(labels)

print(classification_report(labels, predictions))

incorrect_labels = np.concatenate(incorrect_labels)
incorrect_segments = np.concatenate(incorrect_segments)

In [ ]:
predictions = model_upper.logits2probs(model_upper(incorrect_segments))
for label, segment, prediction in zip(
    incorrect_labels, incorrect_segments, predictions
):
    plt.imshow(segment)
    plt.title(
        f"Label: {label} | Pred: {np.argmax(prediction)} "
        f"[{prediction[0]:0.2f}, {prediction[1]:0.2f}]"
    )
    plt.axis("off")
    plt.show()

# Test Models

Load all models

In [ ]:
from hdr_hybrid_butterflies.model import CNNClasifier

model_dict = {}

# upper wing models
for feature_number in range(12):
    name = f"upper_wing_feature_{feature_number:02d}"
    model_dict[name] = CNNClasifier(
        image_size=image_processor.output_dim,
        num_classes=2,
        name=name,
        verbose=False,
    )
    model_dict[name].load_weights(
        os.path.join(
            model_dir, f"upper_model_{feature_number:02d}", "model.weights.h5"
        )
    )

# lower wing models
for feature_number in range(4):
    name = f"lower_wing_feature_{feature_number:02d}"
    model_dict[name] = CNNClasifier(
        image_size=image_processor.output_dim,
        num_classes=2,
        name=name,
        verbose=False,
    )
    model_dict[name].load_weights(
        os.path.join(
            model_dir, f"lower_model_{feature_number:02d}", "model.weights.h5"
        )
    )
model_dict

In [ ]:
model_keys_upper = [
    model_name for model_name in model_dict.keys() if "upper" in model_name
]
model_keys_lower = [
    model_name for model_name in model_dict.keys() if "lower" in model_name
]

predictions_list_upper = []
predictions_list_lower = []
mask_non_hybrid = data_handler.df_meta["hybrid_stat"] == "non-hybrid"
df_meta_non_hybrid = data_handler.df_meta[mask_non_hybrid]
for camid in tqdm(df_meta_non_hybrid["CAMID"], total=np.sum(mask_non_hybrid)):
    df_segments_upper = segment_data_handler.load_df_meta_segments_for_camid(
        camid, labels=["upper"]
    )
    df_segments_lower = segment_data_handler.load_df_meta_segments_for_camid(
        camid, labels=["lower"]
    )

    # if len(df_segments) > 4:
    #     print(df_segments)
    #     break
    if len(df_segments_upper) not in [1, 2]:
        print(f"Camid: {camid} | Upper Segments: {len(df_segments_upper)}")
    if len(df_segments_lower) not in [1, 2]:
        print(f"Camid: {camid} | Lower Segments: {len(df_segments_lower)}")

    if len(df_segments_upper) > 0:
        # load segments for upper
        segments_upper = []
        for idx, row in df_segments_upper.iterrows():
            segment, _ = segment_data_handler.load_by_name(row["filename"])
            segment = image_processor.augment_image(
                segment, apply_augmentations=False
            )
            segments_upper.append(segment)
        segments_upper = np.stack(segments_upper, axis=0)

        # get predictions for upper
        predictions_upper = np.zeros(
            (len(model_keys_upper), len(df_segments_upper), 2)
        )
        for idx, model_name in enumerate(model_keys_upper):
            model = model_dict[model_name]
            logits = model.call(segments_upper)
            predictions_upper[idx] = model.logits2probs(logits)
        predictions_upper = np.max(predictions_upper, axis=1)
    else:
        predictions_upper = np.zeros((len(model_keys_upper), 2))

    if len(df_segments_lower) > 0:
        # load segments for lower
        segments_lower = []
        for idx, row in df_segments_lower.iterrows():
            segment, _ = segment_data_handler.load_by_name(row["filename"])
            segment = image_processor.augment_image(
                segment, apply_augmentations=False
            )
            segments_lower.append(segment)
        segments_lower = np.stack(segments_lower, axis=0)

        # get predictions for lower
        predictions_lower = np.zeros(
            (len(model_keys_lower), len(df_segments_lower), 2)
        )
        for idx, model_name in enumerate(model_keys_lower):
            model = model_dict[model_name]
            logits = model.call(segments_lower)
            predictions_lower[idx] = model.logits2probs(logits)
        predictions_lower = np.max(predictions_lower, axis=1)
    else:
        predictions_lower = np.zeros((len(model_keys_lower), 2))

    predictions_list_upper.append(predictions_upper)
    predictions_list_lower.append(predictions_lower)

predictions_upper = np.stack(predictions_list_upper, axis=0)
predictions_lower = np.stack(predictions_list_lower, axis=0)

In [ ]:
predictions_upper.shape, predictions_lower.shape

In [59]:
from hdr_hybrid_butterflies.data_handler import (
    UpperWingDataHandler,
    LowerWingDataHandler,
)

upper_wing_data_handler = UpperWingDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    image_processor=image_processor,
)

lower_wing_data_handler = LowerWingDataHandler(
    meta_data_path=meta_data_path,
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    image_processor=image_processor,
)

In [ ]:
mask_dict = {}
for i in range(12):
    mask_dict[f"upper_{i:02d}"] = df_meta_non_hybrid["subspecies"].isin(
        upper_wing_data_handler.feature_definitions[i]
    )

for i in range(4):
    mask_dict[f"lower_{i:02d}"] = df_meta_non_hybrid["subspecies"].isin(
        lower_wing_data_handler.feature_definitions[i]
    )

mask_dict.keys()

In [ ]:
predictions_upper.shape

In [ ]:
bins = np.linspace(0, 1, 100)
fig, axes = plt.subplots(4, 3, figsize=(13, 13))

for idx, ax in enumerate(axes.flatten()):
    # ax.hist(
    #     predictions_upper[mask_dict[f"upper_{idx:02d}"], idx, 0],
    #     bins=bins,
    #     histtype="step",
    #     label=f"Upper Wing {idx} | pred 0",
    # )
    # ax.hist(
    #     predictions_upper[~mask_dict[f"upper_{idx:02d}"], idx, 0],
    #     bins=bins,
    #     histtype="step",
    #     label=f"Upper Wing {idx} [non] | pred 0",
    # )
    ax.hist(
        predictions_upper[mask_dict[f"upper_{idx:02d}"], idx, 1],
        bins=bins,
        histtype="step",
        label=f"Upper Wing {idx} | pred 1",
    )
    ax.hist(
        predictions_upper[~mask_dict[f"upper_{idx:02d}"], idx, 1],
        bins=bins,
        histtype="step",
        label=f"Upper Wing {idx} [non] | pred 1",
    )
    ax.set_title(f"Upper Wing | Feature {idx}")
    ax.set_yscale("log")
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Count")
    ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "upper_wing_predictions.png"))

In [ ]:
bins = np.linspace(0, 1, 100)
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for idx, ax in enumerate(axes.flatten()):
    # ax.hist(
    #     predictions_lower[mask_dict[f"lower_{idx:02d}"], idx, 0],
    #     bins=bins,
    #     histtype="step",
    #     label=f"Lower Wing {idx} | pred 0",
    # )
    # ax.hist(
    #     predictions_lower[~mask_dict[f"lower_{idx:02d}"], idx, 0],
    #     bins=bins,
    #     histtype="step",
    #     label=f"Lower Wing {idx} [non] | pred 0",
    # )
    ax.hist(
        predictions_lower[mask_dict[f"lower_{idx:02d}"], idx, 1],
        bins=bins,
        histtype="step",
        label=f"Lower Wing {idx} | pred 1",
    )
    ax.hist(
        predictions_lower[~mask_dict[f"lower_{idx:02d}"], idx, 1],
        bins=bins,
        histtype="step",
        label=f"Lower Wing {idx} [non] | pred 1",
    )
    ax.set_title(f"Lower Wing | Feature {idx}")
    ax.set_yscale("log")
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Count")
    ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "lower_wing_predictions.png"))

In [ ]:
data_handler.df_meta[mask_non_hybrid]